# Synthetic daily feature panel demo
Reuses cleaned synthetic iPHD records, regional attributes and inbound commuting weights. Unknown origin case rates are not zero. Partial pressure is labeled; full pressure remains missing. Onset histories are retrospective and require report-date availability handling before forecasting evaluation. The 7-day sum includes the row date. Future targets must never be model inputs. Matching ZIP to ZCTA assumes equivalence for these demo identifiers. Zero-case dates assume complete synthetic coverage within the observed period.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

# Run this notebook from the project folder. All comments are in English.
PROJECT = Path.cwd()

DATA = Path(r"C:\Users\41240\OneDrive\桌面\MBS\2026 fall\Advanced Analytics\data")

OUTPUT = PROJECT / "daily_panel_demo_outputs"

def build_panel(output=OUTPUT):
    # Reuse the cleaned records from 01_data_and_features.ipynb.
    cases = pd.read_csv(PROJECT / "cleaning_outputs/iphd_cleaned_records.csv",
                        dtype={"ZIP": str, "STATE": str})
    assert pd.to_numeric(cases["IS_SYNTHETIC"]).eq(1).all()
    cases["zcta"] = cases["ZIP"].str.strip().str.zfill(5)
    cases["state"] = cases["STATE"].str.upper().str.strip()
    cases["date"] = pd.to_datetime(cases["ILLNESS_ONSET_DATE"], errors="raise").dt.normalize()
    assert cases["date"].notna().all()
    attrs = pd.read_csv(PROJECT / "regional_attributes_outputs/njny_zcta_regional_attributes_2021.csv",
                        dtype={"zcta": str})
    weights = pd.read_csv(PROJECT / "njny_commute_outputs/njny_commute_zcta_weights_2021.csv",
                          dtype={"origin_zcta": str, "destination_zcta": str})
    assert not attrs.duplicated(["state", "zcta"]).any()

    # The synthetic file is assumed complete only for its eight ZIPs within
    # its observed date range. Absent dates inside this demo range become zero.
    # Real surveillance data requires explicit geographic and date coverage.
    regions = cases[["state", "zcta"]].drop_duplicates().sort_values(["state", "zcta"])
    dates = pd.date_range(cases.date.min(), cases.date.max(), freq="D")
    grid = regions.merge(pd.DataFrame({"date": dates}), how="cross")
    counts = cases.groupby(["state", "zcta", "date"]).size().rename("cases_today").reset_index()
    panel = grid.merge(counts, how="left", on=["state", "zcta", "date"], validate="one_to_one")
    panel["cases_today"] = panel.cases_today.fillna(0).astype("int32")
    panel = panel.merge(attrs, how="left", on=["state", "zcta"], validate="many_to_one")
    assert panel.population.gt(0).all()
    panel = panel.sort_values(["state", "zcta", "date"]).reset_index(drop=True)
    group = panel.groupby(["state", "zcta"], sort=False)
    for lag in [1, 7, 14]:
        panel[f"cases_lag_{lag}"] = group.cases_today.shift(lag)
    panel["cases_sum_7d"] = group.cases_today.transform(lambda s: s.rolling(7, min_periods=7).sum())
    panel["case_rate_today_per_100k"] = panel.cases_today / panel.population * 100000
    panel["case_rate_7d_per_100k"] = panel.cases_sum_7d / panel.population * 100000
    panel["day_of_week"] = panel.date.dt.dayofweek
    panel["month"] = panel.date.dt.month
    for horizon in range(1, 15):
        panel[f"target_cases_t_plus_{horizon}"] = group.cases_today.shift(-horizon)

    # Retain only metro destinations. Keep every original origin and denominator.
    metro_parts = []
    for state in ["nj", "ny"]:
        crosswalk = pd.read_csv(DATA / f"{state}_xwalk.csv.gz",
                                usecols=["stusps", "zcta", "cbsa"], dtype=str)
        metro_parts.append(crosswalk.loc[crosswalk.cbsa.eq("35620"), ["stusps", "zcta"]])
    metro = pd.concat(metro_parts).drop_duplicates().rename(columns={"stusps": "state"})
    destinations = regions.merge(metro, on=["state", "zcta"], validate="one_to_one")
    edges = weights.merge(destinations.rename(columns={"state": "destination_state", "zcta": "destination_zcta"}),
                          on=["destination_state", "destination_zcta"], validate="many_to_one")
    assert np.allclose(edges.groupby(["destination_state", "destination_zcta"]).inbound_weight.sum(), 1)

    # Use only origins with observed synthetic case data. Do not turn other
    # origins into zero-case regions or renormalize their original weights.
    known_edges = edges.merge(regions.rename(columns={"state": "origin_state", "zcta": "origin_zcta"}),
                              on=["origin_state", "origin_zcta"], validate="many_to_one")
    coverage = known_edges.groupby(["destination_state", "destination_zcta"]).inbound_weight.sum()
    rates = panel.pivot(index="date", columns=["state", "zcta"], values="case_rate_today_per_100k")
    rates7 = panel.pivot(index="date", columns=["state", "zcta"], values="case_rate_7d_per_100k")
    pressure_parts = []
    for row in destinations.itertuples(index=False):
        key = (row.state, row.zcta)
        e = known_edges.loc[known_edges.destination_state.eq(row.state) & known_edges.destination_zcta.eq(row.zcta)]
        cov = float(coverage.get(key, 0))
        part = pd.DataFrame({"state": row.state, "zcta": row.zcta, "date": dates,
                             "pressure_source_weight_coverage": cov})
        if len(e):
            columns = pd.MultiIndex.from_frame(e[["origin_state", "origin_zcta"]])
            part["pressure_observed_sources_daily"] = rates.reindex(columns=columns).to_numpy() @ e.inbound_weight.to_numpy()
            part["pressure_observed_sources_7d"] = rates7.reindex(columns=columns).to_numpy() @ e.inbound_weight.to_numpy()
        else:
            part["pressure_observed_sources_daily"] = np.nan
            part["pressure_observed_sources_7d"] = np.nan
        # Full pressure remains missing until all source weights are covered.
        part["commute_weighted_case_rate_daily"] = part.pressure_observed_sources_daily if np.isclose(cov, 1) else np.nan
        part["commute_weighted_case_rate_7d"] = part.pressure_observed_sources_7d if np.isclose(cov, 1) else np.nan
        pressure_parts.append(part)
    pressure = pd.concat(pressure_parts, ignore_index=True)
    result = panel.merge(pressure, on=["state", "zcta", "date"], how="inner", validate="one_to_one")
    result["is_synthetic_demo"] = True
    result["ready_history_and_targets"] = result[["cases_lag_14", "cases_sum_7d", "target_cases_t_plus_14"]].notna().all(axis=1)
    front = ["state", "zcta", "date", "cases_today", "cases_lag_1", "cases_lag_7", "cases_sum_7d",
             "pressure_observed_sources_daily", "pressure_observed_sources_7d", "pressure_source_weight_coverage",
             "commute_weighted_case_rate_daily", "population", "population_density_per_sq_km",
             "target_cases_t_plus_1", "target_cases_t_plus_14"]
    result = result[front + [c for c in result if c not in front]]
    assert not result.duplicated(["state", "zcta", "date"]).any()
    assert panel.cases_today.sum() == len(cases)
    # Check one route-sum independently against the matrix calculation.
    example = result.loc[result.date.eq(dates[14])].iloc[0]
    e = known_edges.loc[known_edges.destination_state.eq(example.state) & known_edges.destination_zcta.eq(example.zcta)]
    source_rates = panel.loc[panel.date.eq(example.date), ["state", "zcta", "case_rate_today_per_100k"]]
    check = e.merge(source_rates, left_on=["origin_state", "origin_zcta"], right_on=["state", "zcta"])
    assert np.isclose((check.inbound_weight * check.case_rate_today_per_100k).sum(), example.pressure_observed_sources_daily)
    output.mkdir(parents=True, exist_ok=True)
    result.to_csv(output / "synthetic_metro_daily_feature_panel.csv", index=False, encoding="utf-8-sig")
    preview = result.loc[result.date.between("2021-01-08", "2021-01-10"), front]
    preview.to_csv(output / "daily_panel_preview.csv", index=False, encoding="utf-8-sig")
    print(f"Cleaned synthetic cases: {len(cases):,}")
    print(f"Covered source ZIPs: {len(regions)}; metro destination ZIPs: {len(destinations)}")
    print(f"Date range: {dates.min().date()} to {dates.max().date()}; days: {len(dates)}")
    print(f"Final rows: {len(result):,}; columns: {len(result.columns)}")
    print("Source weight coverage by destination:")
    print(result.groupby(["state", "zcta"]).pressure_source_weight_coverage.first().to_string())
    print(f"Saved to: {output}")
    return result, preview



In [2]:
result, preview = build_panel()
display(preview)


Cleaned synthetic cases: 14,348
Covered source ZIPs: 8; metro destination ZIPs: 8
Date range: 2020-01-02 to 2023-12-31; days: 1460
Final rows: 11,680; columns: 53
Source weight coverage by destination:
state  zcta 
NJ     07024    0.018757
       07030    0.040056
       07054    0.022594
       07102    0.035768
       07302    0.037615
       08540    0.015827
       08901    0.034616
       08904    0.037879
Saved to: C:\Users\41240\OneDrive\文档\GitHub\practicum-project\Weikun\daily_panel_demo_outputs


,state,zcta,date,cases_today,cases_lag_1,cases_lag_7,cases_sum_7d,pressure_observed_sources_daily,pressure_observed_sources_7d,pressure_source_weight_coverage,commute_weighted_case_rate_daily,population,population_density_per_sq_km,target_cases_t_plus_1,target_cases_t_plus_14
372,NJ,07024,2021-01-08,6,5.0,7.0,31.0,0.173395,1.017062,0.018757,NaN,39701,6100.578956,5.0,6.0
373,NJ,07024,2021-01-09,5,6.0,3.0,33.0,0.137487,1.023093,0.018757,NaN,39701,6100.578956,7.0,3.0
374,NJ,07024,2021-01-10,7,5.0,4.0,36.0,0.097742,0.988282,0.018757,NaN,39701,6100.578956,4.0,3.0
1832,NJ,07030,2021-01-08,5,3.0,5.0,33.0,0.640334,2.459983,0.040056,NaN,59369,18348.014258,6.0,3.0
1833,NJ,07030,2021-01-09,6,5.0,5.0,34.0,0.262233,2.509716,0.040056,NaN,59369,18348.014258,3.0,3.0
1834,NJ,07030,2021-01-10,3,6.0,3.0,34.0,0.288715,2.371355,0.040056,NaN,59369,18348.014258,4.0,7.0
3292,NJ,07054,2021-01-08,0,1.0,3.0,13.0,0.261215,1.409817,0.022594,NaN,30692,856.826861,1.0,1.0
3293,NJ,07054,2021-01-09,1,0.0,4.0,10.0,0.202853,1.477750,0.022594,NaN,30692,856.826861,1.0,2.0
3294,NJ,07054,2021-01-10,1,1.0,1.0,10.0,0.174656,1.429651,0.022594,NaN,30692,856.826861,1.0,0.0
4752,NJ,07102,2021-01-08,2,5.0,4.0,19.0,0.388681,1.949554,0.035768,NaN,13585,4638.640197,3.0,2.0
